Notebook pre-requisites:

In [337]:
!pip install "camelot-py[cv]"
!pip install spacy


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [338]:
!python -m spacy download en_core_web_sm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 23.8 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [339]:
!pip install PyMuPDF
!pip install pdfplumber

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 1. PDF Ingestion & Parsing
- Extract text with page/section anchors (page number, heading hierarchy).
- Preserve structure: titles, subsections, lists, tables, figures’ captions.
- For tables: parse into machine-readable frames (CSV/JSON) when possible.
- Deliverables: raw_text.jsonl (chunks with metadata), tables/*.csv.

In [343]:
import fitz
import json
import re
from pathlib import Path
import shutil
import camelot
import pdfplumber
import pandas as pd

# =============== TOC EXTRACTION ===============
def extract_contents_section(doc):
    toc_lines = []
    toc_started = False
    current_title = ""
    suppressing = False  # True when we've seen THL but not yet seen FOOD

    toc_header_re = re.compile(r"\b(Contents|Table of Contents)\b", re.IGNORECASE)
    toc_entry_re = re.compile(r"\.{3,}\s*(\d+)$")  # dots then page number at EOL
    thl_re = re.compile(r"THL", re.IGNORECASE)
    food_re = re.compile(r"FOOD", re.IGNORECASE)

    for page in doc:
        lines = page.get_text().splitlines()
        for line in lines:
            stripped = line.strip()
            if not stripped:
                continue
            if not toc_started and toc_header_re.search(stripped):
                toc_started = True
                continue
            if not toc_started:
                continue

            # suppression logic
            if suppressing:
                m_food = food_re.search(stripped)
                if m_food:
                    stripped = stripped[m_food.end():].strip()
                    suppressing = False
                    if not stripped:
                        continue
                else:
                    continue

            m_thl = thl_re.search(stripped)
            if m_thl:
                m_food = food_re.search(stripped, m_thl.end())
                if m_food:
                    stripped = stripped[m_food.end():].strip()
                    if not stripped:
                        continue
                else:
                    stripped = stripped[:m_thl.start()].strip()
                    suppressing = True
                    if not stripped:
                        continue

            match = toc_entry_re.search(stripped)
            if match:
                full_title = (current_title + " " + stripped).strip() if current_title else stripped
                toc_lines.append(full_title)
                current_title = ""
            else:
                if current_title:
                    current_title += " " + stripped
                else:
                    current_title = stripped

        last_few = lines[-5:]
        if toc_started and all(not toc_entry_re.search(l.strip()) for l in last_few):
            break

    if current_title:
        if suppressing:
            last_thl = re.search(r"THL", current_title, re.IGNORECASE)
            cleaned_title = current_title[:last_thl.start()].strip() if last_thl else ""
        else:
            cleaned_title = current_title.strip()
        if cleaned_title:
            toc_lines.append(cleaned_title)

    return "\n".join(toc_lines)

# =============== PARSE CONTENTS TO DATAFRAME ===============
def parse_contents_to_df(contents_text):
    lines = [l.strip() for l in contents_text.split("\n") if re.search(r"\d+\s*$", l)]
    rows = []
    for line in lines:
        match = re.match(r"(.+?)\s+(\d+)$", line)
        if match:
            title, page = match.groups()
            title = re.sub(r"\.{2,}", "", title).strip()
            rows.append([title, int(page)])
    df = pd.DataFrame(rows, columns=["title", "page"])
    df = merge_split_rows(df)
    return df

def merge_split_rows(df):
    return df

def normalize_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip().lower()

# =============== TABLE EXTRACTION USING CAMELT/PLUMBER ===============
def extract_tables(pdf_path, output_dir):
    output_dir = Path(output_dir)
    tables_dir = output_dir / "tables"
    if tables_dir.exists():
        shutil.rmtree(tables_dir)
    tables_dir.mkdir(exist_ok=True)

    all_tables = []
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page_str = str(page_num + 1)
        page_tables = []

        # Camelot STREAM
        try:
            tables_stream = camelot.read_pdf(pdf_path, pages=page_str, flavor='stream', edge_tol=50, row_tol=10, strip_text='\n')
            page_tables += [t.df for t in tables_stream if not t.df.empty]
        except Exception:
            pass

        # Camelot LATTICE
        if not page_tables:
            try:
                tables_lattice = camelot.read_pdf(pdf_path, pages=page_str, flavor='lattice', line_scale=40, shift_text=['l','t'])
                page_tables += [t.df for t in tables_lattice if not t.df.empty]
            except Exception:
                pass

        # pdfplumber fallback
        if not page_tables:
            with pdfplumber.open(pdf_path) as pdf:
                page = pdf.pages[page_num]
                plumber_tables = page.extract_tables()
                for pt in plumber_tables:
                    df = pd.DataFrame(pt)
                    page_tables.append(df)

        # Save tables
        for i, df in enumerate(page_tables):
            df = df.fillna("").astype(str).apply(lambda col: col.map(lambda x: re.sub(r"\n", " ", x).strip()))
            df = merge_split_rows(df)
            table_file = tables_dir / f"table_page{page_num+1}_{i+1}.csv"
            df.to_csv(table_file, index=False)
            all_tables.append({
                "table_number": i + 1,
                "page": page_num + 1,
                "file": str(table_file),
                "rows": df.shape[0],
                "columns": df.shape[1]
            })

    print(f"✓ Extracted {len(all_tables)} tables total to '{tables_dir}'")
    return all_tables

# =============== MAIN EXTRACTION FUNCTION ===============
def extract_pdf_with_structure(pdf_path, output_dir="data"):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    doc = fitz.open(pdf_path)
    chunks = []

    # --- Extract TOC and parse ---
    contents_text = extract_contents_section(doc)
    toc_df = parse_contents_to_df(contents_text)
    toc_df = toc_df.sort_values(by="page").reset_index(drop=True)

    toc_index = 0
    current_section = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        lines = page.get_text().splitlines()

        new_sections = []
        while toc_index < len(toc_df) and toc_df.loc[toc_index, "page"] <= page_num + 1:
            new_sections.append(toc_df.loc[toc_index, "title"])
            toc_index += 1

        pending_sections = new_sections.copy()
        buffer = []
        line_idx = 0

        while line_idx < len(lines):
            matched_section = None
            matched_lines_count = 0
            for section_title in pending_sections:
                for lookahead in range(1, min(5, len(lines) - line_idx + 1)):
                    candidate_text = " ".join(lines[line_idx:line_idx+lookahead])
                    if normalize_text(candidate_text).startswith(normalize_text(section_title)):
                        matched_section = section_title
                        matched_lines_count = lookahead
                        break
                if matched_section:
                    break

            if matched_section:
                if current_section and buffer:
                    text_chunk = " ".join(buffer).strip()
                    if text_chunk:
                        chunks.append({"page": page_num+1, "section": current_section, "text": text_chunk})
                current_section = matched_section
                buffer = []
                pending_sections.remove(matched_section)
                line_idx += matched_lines_count
                continue
            else:
                if current_section:
                    buffer.append(lines[line_idx].strip())
                line_idx += 1

        if current_section and buffer:
            text_chunk = " ".join(buffer).strip()
            if text_chunk:
                chunks.append({"page": page_num+1, "section": current_section, "text": text_chunk})

        if pending_sections:
            for missing_section in pending_sections:
                print(f"[WARNING] Section '{missing_section}' expected on page {page_num+1} but not found.")
                if buffer:
                    text_chunk = " ".join(buffer).strip()
                    if text_chunk:
                        chunks.append({"page": page_num+1, "section": missing_section, "text": text_chunk})
                        buffer = []

    # Save text
    output_file = output_dir / "raw_text.jsonl"
    with open(output_file, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    print(f"✓ Extracted {len(chunks)} text chunks")
    print(f"✓ Clean text saved to {output_file}")

    # --- Extract tables using Camelot/pdfplumber ---
    tables_info = extract_tables(pdf_path, output_dir)

    return chunks, toc_df, tables_info


In [344]:
pdf_path = "data/sustainable-health-from-food_web.pdf"
chunks, toc_df, tables = extract_pdf_with_structure(pdf_path)

[WARNING] Section 'Appendix 8. Recommended intakes of fat, carbohydrates and protein for adults and children over 2 years (without alcohol and with fibre taken into account)' expected on page 102 but not found.
✓ Extracted 134 text chunks
✓ Clean text saved to data/raw_text.jsonl


/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (0, 0, 498.898, 708.661)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (63.7008, 137.1649, 437.67889999999966, 681.5164555555556)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (92.0472, 302.90479999999997, 462.89330000000035, 684.5557947368421)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.

✓ Extracted 137 tables total to 'data/tables'


/Users/beatricezani/Desktop/Uni/NLP/pdf_to_KnowledgeGraph_LLM/.venv/lib/python3.10/site-packages/camelot/parsers/base.py:238: UserWarning: No tables found in table area (53.76729999999999, 246.3673, 449.4282999999997, 630.1827161904762)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


## 3) NER & Keyphrase Extraction

In [345]:
!pip install spacy
!pip install transformers
!pip install keybert
!pip install scispacy
!pip install fuzzywuzzy python-Levenshtein
!python -m spacy download en_core_web_sm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached scispacy-0.6.2-py3-none-any.whl.metadata (20 kB)
  Using cached conllu-6.0.0-py3-none-any.whl.metadata (21 kB)
  Using cached numpy-1.26.4-cp310-cp310-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached nmslib-metabrainz-2.1.3.tar.gz (196 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pysbd-0.3.4-py3-none-any.whl.metadata (6.1 kB)
  Using cached pybind11-3.0.1-py3-none-any.whl.metadata (10.0 kB)
INFO: pip is looking at multiple versions of thinc to determine which version is compatible with other requirements. This could take a while.
  Using cached thinc-8.3.4-cp310-cp310-macosx_11_0_arm64.whl.metadata (15 kB)
  Using cached blis-1.2.1-cp310-cp310-macosx_11_0_arm64.whl.metadata (7.4 kB)
Using cached scispacy-0.6.2-py3-none-any.whl (62 kB)
Using cached numpy-1.26.4-cp310-cp310-macosx_11_0_arm64.whl (14.0 MB)
Using cached 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [347]:
import json
import pandas as pd
import spacy
from keybert import KeyBERT
import yaml

# load Text Chunks
with open('data/raw_text.jsonl', 'r') as f:
    text_chunks = [json.loads(line) for line in f]
df_chunks = pd.DataFrame(text_chunks)
print(f"Loaded {len(df_chunks)} text chunks.")

# Load Ontology
with open('data/ontology.yaml', 'r') as f:
    ontology = yaml.safe_load(f)
entity_classes = list(ontology.get('classes', {}).keys())
print(f"Ontology classes: {entity_classes}")

Loaded 134 text chunks.
Ontology classes: ['ingredient', 'nutrient', 'technique', 'dietaryGuideline', 'healthOutcome', 'environmentImpact']
